# Interpreting Aging Biology Data with LFM2 Longevity Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Liquid4All/cookbook/blob/main/guides/lfm2_longevity.ipynb)

Aging data comes in formats that rarely match: clinical panels, DNA methylation arrays, transcriptomic profiles, plasma proteomics, and curated genetic evidence. Aging clocks and biological foundation models are typically built for one modality at a time. 

[LFM2-1.2B-Longevity](https://huggingface.co/LiquidAI/LFM2-1.2B-Longevity) and [LFM2-2.6B-Longevity](https://huggingface.co/LiquidAI/LFM2-2.6B-Longevity) read all of them as structured text. Each is a full-parameter supervised fine-tune of the corresponding LFM2 base model on aging-related multi-omics and clinical data, developed jointly by [Liquid AI](https://www.liquid.ai) and [Insilico Medicine](https://insilico.com) alongside the *Cell* paper [An Open Benchmark and Language Models for AI in Aging Biology](https://doi.org/10.1016/j.cell.2026.08.026) and the [LongevityBench](https://huggingface.co/datasets/insilicomedicine/longebench) dataset. Our [release blog post](https://www.liquid.ai/blog/longevity) covers the benchmark design and results.

> Outputs are model predictions, not clinical advice, and should be validated experimentally.

## What LongevityBench measures

LongevityBench comprises 17 tasks and 25,457 prompts across five biodata domains, each pairing structured biological data with one of four answer formats. Most domains appear in more than one format, which isolates the effect of the requested output while holding the underlying data fixed.

| Domain | Source | Pairwise | Multiclass | Regression | Binary |
|---|---|---|---|---|---|
| Clinical | NHANES | age, mortality | age, mortality | age | mortality |
| DNA methylation | GEO | age | age | age | — |
| Transcriptomics | GTEx | age | age | — | — |
| Proteomics | Olink | age | age | age | — |
| Genetics | OpenGenes, SynergyAge | lifespan | — | lifespan | expression direction |

Every evaluation task is separated from its training counterpart by a split defined on the underlying biology, such as survey wave for NHANES or study of origin for methylation. The examples below are drawn from the held-out data.

## Setup

The Longevity models are supported natively in Transformers from version 5.1.0 onward, so no custom modeling code is required.

In [1]:
%pip install -q -U "transformers>=5.1.0" torch accelerate

## Load the model

Both models run on a free Colab instance or a laptop, and share the same interface, so switching `model_id` is the only change needed to run the other.

In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer

lfm_models = [
    "LiquidAI/LFM2-1.2B-Longevity",
    "LiquidAI/LFM2-2.6B-Longevity",
]

model_id = "LiquidAI/LFM2-1.2B-Longevity"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    dtype="bfloat16",
)

## Task examples

Training prompts used a dynamic-thinking template, where the user turn ends in `/think` or `/no_think` to select the response mode. Every prompt below ends in `/no_think`, matching the published evaluation. Without it the model may enter thinking mode and exhaust the token budget before answering.

### Clinical records (NHANES)

This task (`LB-0034`) renders two NHANES participants as demographics, body measurements, and around twenty blood markers, then asks which is older. The profiles are not harmonized: glucose is reported in mg/dL for A and mmol/L for B, and creatinine also switches units, so the model has to read the units rather than compare the numbers directly.

The expected answer for the example below is **A**.

In [3]:
prompt = """Which participant is older, based on their clinical blood test results? Answer with one letter.

Options: A. Participant A  B. Participant B

Participant A:
Demographics: Female.
Body measurements: BMI 24.3 kg/m2.
Blood panel: Albumin (g/L) 49.00; Alkaline phosphatase (U/L) 91.00; Blood urea nitrogen (mg/dL) 8.00; Creatinine (umol/L) 48.82; Serum glucose (mg/dL) 180.00; Total bilirubin (mg/dL) 0.50; Uric acid (mg/dL) 4.60; Total cholesterol (mg/dL) 295.00; HDL (mg/dL) 105.00; LDL (mg/dL) 172.00; Triglyceride (mg/dL) 90.00; C-reactive protein (mg/dL) 0.28; White blood cell count (1000 cells/uL) 7.50; Red blood cell count (million cells/uL) 4.06; Mean cell volume (fL) 98.20; Red cell distribution width (%) 11.40; Glycohemoglobin (%) 8.40; Vitamin A (umol/L) 2.78; Vitamin E (umol/L) 70.67; Vitamin B12 (pmol/L) 811.80; Bone alkaline phosphatase (ug/L) 10.05; Cadmium (nmol/L) 13.35.

Participant B:
Demographics: Female.
Biometrics: BMI 27.1 kg/m2.
Blood test results: Albumin (g/L) 46.00; Alkaline phosphatase (U/L) 64.00; Blood urea nitrogen (mg/dL) 9.00; Creatinine (mg/dL) 0.65; Serum glucose (mmol/L) 4.22; Total bilirubin (mg/dL) 0.40; Uric acid (mg/dL) 4.00; Total cholesterol (mg/dL) 213.00; HDL (mg/dL) 68.00; C-reactive protein (mg/dL) 0.17; White blood cell count (1000 cells/uL) 8.00; Red blood cell count (million cells/uL) 3.93; Mean cell volume (fL) 95.30; Red cell distribution width (%) 11.90; Segmented neutrophils percent (%) 48.60; Lymphocyte percent (%) 35.30; Monocyte percent (%) 12.30; Eosinophils percent (%) 2.20; Basophils percent (%) 1.60; Glycohemoglobin (%) 5.40; Vitamin A (umol/L) 2.55; Vitamin E (umol/L) 54.18; Vitamin B12 (pmol/L) 296.68; Bone alkaline phosphatase (ug/L) 15.24; Cadmium (nmol/L) 5.34. /no_think"""

input_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
    return_dict=False,
).to(model.device)

output = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.3,
    min_p=0.15,
    repetition_penalty=1.05,
    max_new_tokens=512,
)

print(tokenizer.decode(output[0, input_ids.shape[-1]:], skip_special_tokens=True))

A


**Adapting this to other task types.**

* **Multiclass age (`LB-0030`)**: one profile, assigned to an age decade.

  ```
  Which age group does this participant belong to? Answer with one letter.

  Options: A. 20-29  B. 30-39  C. 40-49  D. 50-59  E. 60-69  F. 70-79  G. 80+ /no_think
  ```

* **Regression age (`LB-0038`)**: one profile, no options.

  ```
  How old is this participant? Answer with a number of years. /no_think
  ```

* **Binary mortality (`LB-0042`)**: one profile, survival status at follow-up.

  ```
  Is this participant alive or deceased at follow-up? Answer with one letter.

  Options: A. Alive  B. Deceased /no_think
  ```

* **Pairwise mortality (`LB-0050`)**: two profiles, as in the example above, asking which participant survived longer.
* **Multiclass mortality (`LB-0046`)**: one profile, with survival time bucketed into the `Options:` block.

### DNA methylation (GEO)

Methylation underlies the best-known aging clocks. Each CpG site in this task (`LB-0006`) carries its chromosomal position, island context, nearest gene, GO annotations, and the beta values for both individuals.

The full prompt lists around 160 CpG sites at a median of 21,330 tokens; the version below is abbreviated to the eight largest differences.

The expected answer for the example below is **B**.

In [4]:
prompt = """Based on the following DNA methylation profiles from blood samples, which individual is older?

Options: A: A female, from Canada, white, BMI 38.3, non-smoker. ; B: A female, from Germany, diagnosed with osteoporosis.

Methylation data:
The following CpG sites show the largest methylation differences between the two individuals: cg15720535 (chr9:136688133, -, CpG Island). In AGPAT2 (1-acylglycerol-3-phosphate O-acyltransferase 2) promoter, 676bp upstream of TSS; ENSG00000301427 promoter, 85bp upstream of TSS. Process: positive regulation of cytokine-mediated signaling pathway, CDP-diacylglycerol biosynthetic process, response to xenobiotic stimulus, phospholipid metabolic process. Function: 1-acylglycerol-3-phosphate O-acyltransferase activity, acyltransferase activity, transferase activity. Component: endoplasmic reticulum membrane, specific granule membrane, plasma membrane, endoplasmic reticulum. A: 0.15, B: 0.68, B much higher. | cg02828104 (chr20:50154263, -, S_Shore 91bp). In PEDS1 (plasmanylethanolamine desaturase 1) promoter, 509bp upstream of TSS; PEDS1-UBE2V1 (PEDS1-UBE2V1 readthrough) promoter, 626bp upstream of TSS. Process: ether lipid biosynthetic process, fatty acid metabolic process, lipid metabolic process. Function: oxidoreductase activity, plasmanylethanolamine desaturase activity, protein binding. Component: endoplasmic reticulum, endoplasmic reticulum membrane. A: 0.15, B: 0.64, B much higher. | cg18680834 (chr19:30372442, +, N_Shelf 2335bp). In ZNF536 (zinc finger protein 536) body (exon_1), 148kbp downstream of TSS. Process: negative regulation of transcription by RNA polymerase II, regulation of DNA-templated transcription. Function: DNA-binding transcription repressor activity, RNA polymerase II-specific, RNA polymerase II cis-regulatory region sequence-specific DNA binding, metal ion binding, DNA-binding transcription factor activity, RNA polymerase II-specific. Component: nucleus. A: 0.27, B: 0.75, B much higher. | cg18059933 (chr8:94950235, +, S_Shore 91bp). In TP53INP1 (tumor protein p53 inducible nuclear protein 1) promoter, 840bp upstream of TSS; NDUFAF6 (NADH:ubiquinone oxidoreductase complex assembly factor 6) body, 55kbp downstream of TSS. Process: negative regulation of cell population proliferation, positive regulation of DNA-templated transcription, cellular oxidant detoxification, autophagosome assembly. Function: antioxidant activity, protein binding. Component: cytoplasmic vesicle, autophagosome, nucleoplasm, PML body. Process: mitochondrial respiratory chain complex I assembly. Function: protein binding. Component: cytoplasm, mitochondrial inner membrane, mitochondrion, nucleus. A: 0.23, B: 0.69, B much higher. | cg19403023 (chr16:2798796, -, CpG Island). In PRSS41 (serine protease 41) body (intron_2), 326bp downstream of TSS. Process: protein processing, proteolysis. Function: hydrolase activity, peptidase activity, serine-type endopeptidase activity, serine-type peptidase activity. Component: extracellular region, intracellular organelle, plasma membrane, side of membrane. A: 0.23, B: 0.69, B much higher. | cg10591174 (chr12:110920642, -, OpenSea). In MYL2 (myosin light chain 2) body, 132kbp downstream of TSS. Process: positive regulation of the force of heart contraction, ventricular cardiac muscle tissue morphogenesis, negative regulation of cell growth, muscle cell fate specification. Function: structural constituent of muscle, myosin heavy chain binding, actin monomer binding, calcium ion binding. Component: actin cytoskeleton, cardiac myofibril, myosin complex, cytoplasm. A: 0.17, B: 0.62, B much higher. | cg24949488 (chr10:96304605, -, OpenSea). In DNTT (DNA nucleotidylexotransferase) body (exon_1), 196bp downstream of TSS; ENSG00000229418 body (intron_1), 2kbp downstream of TSS. Process: double-strand break repair via nonhomologous end joining, DNA biosynthetic process, DNA modification, DNA repair. Function: DNA nucleotidylexotransferase activity, nucleotidyltransferase activity, transferase activity, metal ion binding. Component: cytosol, nucleoplasm, nucleus. A: 0.39, B: 0.81, B much higher. | cg25165880 (chr1:6394266, +, S_Shore 141bp). In ACOT7 (acyl-CoA thioesterase 7) promoter, 499bp upstream of TSS; ENSG00000271746 body (exon_1), 711bp downstream of TSS. Process: medium-chain fatty-acyl-CoA catabolic process, medium-chain fatty acid biosynthetic process, coenzyme A biosynthetic process, acyl-CoA metabolic process. Function: long-chain fatty acyl-CoA hydrolase activity, carboxylic ester hydrolase activity, protein homodimerization activity, fatty-acyl-CoA binding. Component: cytoplasm, cytosol, extracellular exosome, mitochondrion. A: 0.22, B: 0.64, B much higher. /no_think"""

input_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
    return_dict=False,
).to(model.device)

output = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.3,
    min_p=0.15,
    repetition_penalty=1.05,
    max_new_tokens=512,
)

print(tokenizer.decode(output[0, input_ids.shape[-1]:], skip_special_tokens=True))

B


**Adapting this to other task types.** 

* **Multiclass age (`LB-0002`)**: one profile, assigned to an age decade.

  ```
  Which age group is this individual in? Answer with one letter.

  Options: A. 20-29  B. 30-39  C. 40-49  D. 50-59  E. 60-69  F. 70-79  G. 80+ /no_think
  ```

* **Regression age (`LB-0010`)**: one profile, no options.

  ```
  How old is this individual? Answer with a number of years. /no_think
  ```

### Transcriptomics (GTEx)

This task (`LB-0018`) compares two GTEx adrenal gland samples, each presented as a rank-ordered list of the most expressed genes plus a GSEA comparison of pathway enrichment scores. The ranking encodes relative order, not expression magnitude.

The expected answer for the example below is **A**. Pairwise concordance on this task is the lowest in the benchmark and close to chance, so the answer can flip between runs.

In [5]:
prompt = """Which of the following two Adrenal Gland samples comes from an older individual?

Options: A. Sample A  B. Sample B

Sample a: Demographics: male
Transcriptomic profile:
Top-2774 expressed genes: COX1; ND4; CYP17A1; ATP6; CYTB; COX3; ND2; COX2; CYP11B1; STAR; ND1; DHCR24; MEG3; DLK1; HSD3B2; ATP8; SCARB1; CYP21A2; ND5; AKR1B1; ND4L; ND3; EEF1A1; HSP90AA1; PEBP1; HSPA8; NEAT1; CLU; EEF2; APOE; ASH2L; PSAP; TPT1; DDX5; CTSD; HSP90AB1; CYP11A1; GAPDH; EPHX1; ACTB; FTH1; FTL; ACADVL; SERPINA5; FADS2; PEG10; ND6; ACTG1; BEST1; SPARC; UBC; HSPD1; ALDOA; SRRM2; AOX1; GSTA1; PKM; NOV; RARRES2; SQSTM1; RPL3; MGST1; UBB; B2M; SCD; DDX17; RPS8; TBX3; ABCC3; ATP5B; FDPS; RPS6; CYB561A3; GNAS; C4B; MYL9; C3; TMBIM6; HSPB1; FDX1; H19; SDC3; RPL13A; HNRNPA2B1; RHOB; RPLP0; GSN; TMSB4X; RACK1; ATP5A1; FDXR; GJA1; CALM2; BORCS7; APP; P4HB; HSPA1A; RPL13; BSG; RPL8; TNXB; RPL13AP5; DCN; DDB1; LAMB2; SREBF1; KCNK3; HSPA1B; RPL4; ALAS1; RPLP1; CTSA; SLC25A6; FAM198B; RPL5; AP2M1; MALAT1; AS3MT; VAT1; HSPA9; RDX; TUBB4A; NR5A1; MGP; C5orf45; OAZ1; SEMA3B; GNS; EIF4G2; RPL19; RPL7A; LDLR; EIF4A2; PDK4; HLA-A; C4A; HSPG2; SLC25A3; RPS18; MYL6; VIM; RPL10; C2CD2; RPS11; TXNIP; IDH1; TKFC; RPS3; FDFT1; PCBP2; LONP1; C7; GSTA4; ATN1; CYB5B; TIMP1; PLBD2; LSS; NDRG2; HLA-C; SREBF2; HNRNPH1; NFE2L1; RPL15; ALDH2; SNRNP70; CERS2; LGALS3BP; CD46; MATR3; RPS4X; ASPH; ITM2B; LDHB; HMGCS1; FOSL2; RPL18; RPS9; SOAT1; HDLBP; GPI; PRDX2; ALDH3A2; RBBP7; EZR; RHOA; SERF2; CYP21A1P; RPL28; MSMO1; LENG8; ARHGEF40; POR; RPS24; PAPSS2; FLNA; PRDX3; EIF4G1; DDIT4; NACA; HNRNPU; SPTBN1; AES; RPL37A; TM7SF2; RPL23; PLXNB2; ENO1; LRRC75A-AS1; RTN4; VWA5B2; SYNPO; TSC22D3; RPS19; CAPN2; VCP; GRAMD1B; APOC1; ACTN4; RPL11; MYH9; RPL17; STAB1; LDHA; PABPC1; TIMP3; PCBP1; CYB5R3; SENP3; FAM166B; RPL29; CCND3; SARAF; TPI1; SLC40A1; FADS1; CTNNB1; RPS14; UBA1; FASN; CTSB; TLN1; RPS5; DDX39B; MBOAT7; EIF4A1; RPLP2; ADGRV1; RPS12; CTNNA1; RPL36; HNRNPK; TKT; EIF1; HSP90B1; ECE1; BGN; GOLGA8A; MLEC; PPIF; CLTC; SCD5; ATF4; AHSA2; ABCB1; WBP2; CTTN; MTCH1; NAP1L1; HIF1A; POLR2A; ECH1; SF3B1; SORBS2; SRSF5; CDKN1C; CD63; TNS1; ANXA5; NCL; HMGCR; DAB2; DDR2; NUMA1; RPS2; PIK3R2; TXNRD1; RPL27; HADHB; RPS20; CCNL2; MCFD2; SOD1; ZNF275; TPP1; PPP2R1A; KIAA1522; PEG3; TNS2; CSDE1; RBM39; RAB31; RPS17; MT2A; PGRMC1; ERGIC1; WDR6; COL18A1; STAT6; EEF1D; RPL32; ABLIM1; NDRG4; EPB41L1; MAGED1; RNF213; EIF4B; PGK1; COQ8A; MVP; SRSF6; IDI1; PINK1; SGK1; FSTL1; GABARAP; YWHAZ; CNN3; RPL31; COL3A1; LSP1; PLEC; MKNK2; MYL12B; HNRNPC; PTOV1; EBP; CIRBP; AP2A1; ZBED6CL; EEF1G; FLOT1; RPL30; DNAJA1; A2M; YWHAE; RGN; CANX; COL6A1; ALAD; STIP1; RAP1GAP; MRFAP1; HSPB6; DDX24; PRPF8; FAM129B; SLC16A9; RPL10A; CCT3; UBE2D3; RPL35A; DYNC1H1; CHCHD2; GHITM; AHNAK; EWSR1; RPS16; SERINC1; SCAP; BRE; VAPA; ARF1; BST2; MORF4L2; ELOVL5; TPD52L1; LAPTM4A; SEPT9; UBA52; SRCAP; HP1BP3; RPL27A; SON; USP22; PCBP1-AS1; PRDX5; CFL1; MIF; CNBP; RPS15A; NONO; RPS23; APLP2; MDH2; RPL37; GPX3; PBXIP1; MAT2A; CS

Sample b: Demographics: female
Transcriptomic profile:
Top-2751 expressed genes: COX1; ND4; STAR; ATP6; CYP17A1; ND1; COX3; CYP11B1; ND2; COX2; CYTB; ND5; ATP8; ND6; SCARB1; CYP21A2; ACTB; GAPDH; ASH2L; HSD3B2; DHCR24; EEF1A1; ND4L; NEAT1; ND3; CYP11A1; HSP90AB1; AKR1B1; EEF2; ACTG1; CTSD; MEG3; PSAP; P4HB; TPT1; PEG10; UBC; HSPA8; PKM; HSPA9; HSPA5; SPARC; DLK1; H19; ENO1; RPL8; HSPD1; ALDOA; HSP90B1; CLU; FTH1; FLNA; SREBF1; FTL; APOE; CYP21A1P; ATP5B; UBB; CD63; KCNK3; MYL6; B2M; GNAS; SRRM2; PEBP1; C7; RPS6; TNXB; ACADVL; RTN4; DDX5; RPS8; RHOA; RPL3; RPS18; HSP90AA1; SQSTM1; RPLP0; RACK1; LDHA; ALAS1; TIMP1; GJA1; AP2M1; RPL4; FDX1; XIST; BEST1; APP; TUBB; RPL19; MGST1; RPL13AP5; CANX; GNS; TPI1; SLC25A6; RPL13; VCP; CALR; TMBIM6; SEMA3B; MYH9; RPL7A; CFL1; TUBB4A; HNRNPA2B1; TXNRD1; HIF1A; MT2A; CTSA; RPLP2; MYL9; MALAT1; BSG; RPS3; ABCB1; SPTBN1; TUBA1B; FADS2; EPHX1; EIF4A1; SENP3; SCD; BORCS7; OAZ1; ATP5A1; EIF4G1; PLBD2; RDX; RPLP1; RPL13A; ACTN4; CTSB; HDLBP; RPS11; AOX1; LONP1; RPL28; RPL37A; GPI; SLC25A3; DDX17; POR; AS3MT; CALM2; ABCC3; RPS4X; RPS5; NCL; RPS2; VIM; RHOB; EPB41L1; ANXA2; PSMD2; RPL10; CTNNA1; C3; EZR; HSPB1; HNRNPK; CLTC; GSN; HLA-C; RPS24; TSIX; RPS12; MRFAP1; PFN1; TMSB4X; SEC61A1; SYNPO; EIF4G2; DDR2; TLN1; RPS17; PRPF8; CIRBP; IGFBP4; ECE1; LSP1; RPN2; DCN; ALDH3A2; CAPN2; VAT1; C5orf45; C4B; SLC3A2; FAM129B; RPL5; TBX3; DDB1; RPL11; SERF2; PPP2R1A; HSPG2; GRAMD1B; ARF1; RPL32; MIF; RPL23; FDXR; DDX39B; AADAC; PGK1; RBBP7; NFE2L1; MVP; YWHAE; RPS19; RPL27; ATP1A1; HLA-A; FOSL2; EIF1; UBA1; RPL31; RPS9; EIF5A; ARHGDIA; ASPH; RPL36; PPIB; UBA52; ZNF275; EEF1D; DYNC1H1; RPS14; IGFBP5; MATR3; PRDX2; RRBP1; RPL17; CHCHD2; CYP11B2; CACNA1H; SPTAN1; RPL15; EEF1G; TXNIP; AARS; NACA; RAN; RPS16; MYL12B; BGN; LMNA; COL1A1; YWHAG; RPL18; COL3A1; PABPC1; MTCH1; XRCC6; RPL37; MCFD2; PAPSS2; HLA-B; RPL29; COL4A2; PCBP1; PLEC; RARRES2; CSDE1; EWSR1; FLOT1; FAM166B; SLC47A1; YWHAZ; ERGIC1; SDC3; IFITM3; PCBP2; RPL30; SON; SARAF; SCAP; PPIF; CNN3; ATN1; ATF4; NR5A1; PDIA6; ENG; RAB7A; ILF3; LDLR; PPIA; THBS1; HSPA1B; ELOVL5; TIMP3; CYB5B; LGALS1; RPL10A; ANXA5; GPX3; SND1; RPS20; PDIA3; RPL27A; NME1-NME2; HUWE1; YWHAQ; LDHB; HNRNPU; PTMA; LENG8; YBX1; GHITM; AP2A1; HSPA1A; RPS27A; VWA5B2; DNER; GANAB; CLPTM1; GALNT2; MGP; WDR1; MLEC; CYB5R3; DDX24; MDH2; HDAC7; PRDX1; IARS; SERPINA5; PEG3; HNRNPH1; SLC23A2; TAGLN2; COL1A2; LGALS3BP; RPL26; RPS23; RPL35; AHNAK; SURF4; SERINC1; COL4A1; NUMA1; TNS1; YWHAB; ABCA1; LRRC59; APLP2; SSR2; NOV; AES; NUDC; TKT; CERS2; DDOST; NFIC; FSTL3; CCT7; ATP6V0C; BCAP31; C4A; CS; SLC16A9; MORF4L2; SOD2; C1R; CD151; PLEKHB2; SOAT1; TPM3; PEA15; TCIRG1; PLTP; PGRMC1; TPP1; RPS25; C2CD2; DDX3X; RPL41; LRRC75A-AS1; H3F3B; FLII; PSMA7; EIF3A; RPL14; COPG1; MSN; POLR2A; SEPT9; GNB1; GRHPR; VAPA; KCNQ1; ITM2B; FAU; TMED10; HM13; EDF1; GABARAP

Gene set enrichment analysis:
Sample-A shows low activity in Phospholipase C Mediated Cascade Fgfr4 (NES: -0.07, genes: FGF4, FGF6, FGF16, FGF19, FGF23 and 10 other genes), while Sample-B shows low activity (NES: -0.13, genes: FGF16, FGF19, FGF4, FGF6, FGF8 and 10 other genes). Sample-A shows very high activity in Cholesterol Biosynthesis (NES: 0.46, genes: GGPS1, ARV1, LBR, PLPP6, IDI2 and 22 other genes), while Sample-B shows moderately high activity (NES: 0.40, genes: PLPP6, HSD17B7, ARV1, LBR, IDI2 and 22 other genes). Sample-A shows low activity in Defective C1Galt1C1 Causes Tnps (NES: -0.14, genes: MUC17, MUC16, MUC5B, MUC7, MUC15 and 12 other genes), while Sample-B shows low activity (NES: -0.00, genes: MUC15, MUCL1, MUC16, MUC17, MUC12 and 12 other genes). Sample-A shows very low activity in Defective Galnt3 Causes Hftc (NES: -0.18, genes: MUC17, MUC16, MUC5B, MUC7, MUC15 and 11 other genes), while Sample-B shows low activity (NES: -0.05, genes: MUC15, MUCL1, MUC16, MUC17, MUC12 and 11 other genes). /no_think"""

input_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
    return_dict=False,
).to(model.device)

output = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.3,
    min_p=0.15,
    repetition_penalty=1.05,
    max_new_tokens=512,
)

print(tokenizer.decode(output[0, input_ids.shape[-1]:], skip_special_tokens=True))

A


**Adapting this to other task types.**

* **Multiclass age (`LB-0014`)**: one ranked profile, assigned to an age decade.

  ```
  Which age group is this Adrenal Gland sample from? Answer with one letter.

  Options: A. 20-29  B. 30-39  C. 40-49  D. 50-59  E. 60-69  F. 70-79  G. 80+ /no_think
  ```

### Proteomics (Olink)

The Olink Explore 3072 assay reports 2,890 plasma proteins across four panels. Rather than listing all of them, this task (`LB-0299`) reports the 100 most differentially abundant between the two individuals, each as a pair of NPX values on a log2 scale.

The expected answer for the example below is **B**.

In [6]:
prompt = """Measurements were performed using Olink Explore 3072 proximity extension assay (PEA) plasma proteomics. Values are NPX (Normalized Protein eXpression, log2 scale). 2890 proteins measured across Cardiometabolic, Inflammation, Oncology, and Neurology panels.

Based on their plasma proteome profiles, which individual is older?

Proteomic data:
Top-100 differentially abundant proteins out of 2890 measured (NPX): TMED1 (A: 10.44, B: -8.38, A much higher); PTN (A: -6.18, B: 4.14, B much higher); AGBL2 (A: -8.37, B: 1.93, B much higher); DDX4 (A: -9.62, B: -0.29, B much higher); KIAA1549 (A: -10.31, B: -1.06, B much higher); CNTF (A: -7.94, B: 1.24, B much higher); ZP3 (A: -5.94, B: 3.07, B much higher); RAD51 (A: -8.74, B: 0.25, B much higher); OSTN (A: -8.24, B: 0.07, B much higher); RRAS (A: -8.45, B: -0.31, B much higher); SERPINB5 (A: 1.43, B: -6.64, A much higher); SLC1A4 (A: -7.99, B: 0.00, B much higher); FMR1 (A: 0.70, B: -7.07, A much higher); IL3 (A: -5.98, B: 1.32, B much higher); PNMA2 (A: -8.67, B: -1.39, B much higher); TMCO5A (A: -7.34, B: -0.18, B much higher); MORF4L1 (A: -8.36, B: -1.26, B much higher); CSH1 (A: -1.69, B: -8.52, A much higher); BLOC1S2 (A: 0.78, B: -5.90, A much higher); EPN1 (A: -0.77, B: 5.45, B much higher); IL18RAP (A: -1.02, B: 5.14, B much higher); PARD3 (A: -0.98, B: -7.08, A much higher); KRT17 (A: -1.32, B: -7.40, A much higher); HDAC9 (A: -6.97, B: -0.98, B much higher); MCEE (A: -0.73, B: 5.05, B much higher); PYY (A: -1.59, B: 4.19, B much higher); CYP24A1 (A: -0.70, B: 5.02, B much higher); TDGF1 (A: 4.07, B: -1.55, A much higher); PPP1R12B (A: -5.86, B: -0.23, B much higher); LELP1 (A: -0.72, B: 4.87, B much higher); DAND5 (A: -1.18, B: 4.39, B much higher); RRP15 (A: 0.71, B: -4.73, A much higher); NPPB (A: -3.74, B: 1.31, B much higher); GCG (A: -1.61, B: 3.39, B much higher); ZPR1 (A: -1.35, B: 3.54, B much higher); IL1RN (A: -0.89, B: 3.99, B much higher); KIR2DS4 (A: 3.83, B: -1.01, A much higher); PHLDB2 (A: 3.92, B: -0.87, A much higher); CCND2 (A: -4.27, B: 0.33, B much higher); PM20D1 (A: 1.18, B: -3.34, A much higher); WASL (A: -0.33, B: 4.12, B much higher); TRIM40 (A: -2.79, B: -7.19, A much higher); KIF20B (A: 2.39, B: -1.93, A much higher); SERPINH1 (A: -3.64, B: -7.94, A much higher); METAP1D (A: 3.73, B: -0.55, A much higher); FGF9 (A: 2.80, B: -1.42, A much higher); PLB1 (A: 2.94, B: -1.20, A much higher); VCPKMT (A: -0.33, B: 3.77, B much higher); CENPJ (A: -4.13, B: -8.21, A much higher); HIF1A (A: -0.82, B: 3.25, B much higher); GSTT2B (A: -3.39, B: 0.66, B much higher); HADH (A: -0.44, B: 3.56, B much higher); APOL1 (A: 3.07, B: -0.87, A much higher); NGRN (A: 3.88, B: 0.02, A much higher); MUC16 (A: -3.87, B: -0.03, B much higher); CRYM (A: -0.55, B: 3.25, B much higher); TSPAN7 (A: -1.57, B: 2.18, B much higher); CHGA (A: -2.05, B: 1.63, B much higher); CKMT1A_CKMT1B (A: -2.18, B: 1.43, B much higher); SPRR1B (A: -1.03, B: -4.65, A much higher); ALPP (A: 0.64, B: -2.83, A much higher); CILP (A: -1.75, B: 1.71, B much higher); ESR1 (A: -1.86, B: 1.60, B much higher); PDCL2 (A: -2.42, B: 1.01, B much higher); IL36G (A: -0.24, B: 3.14, B much higher); GDF15 (A: -0.81, B: 2.55, B much higher); ACRV1 (A: -0.90, B: 2.44, B much higher); HEBP1 (A: 2.99, B: -0.33, A much higher); TFAP2A (A: -0.63, B: 2.68, B much higher); REG4 (A: -1.13, B: 2.16, B much higher); GADD45B (A: -2.06, B: 1.15, B much higher); PCARE (A: -0.97, B: 2.24, B much higher); TRDMT1 (A: -2.47, B: 0.69, B much higher); LY6D (A: -0.30, B: 2.85, B much higher); PFDN2 (A: 1.22, B: -1.93, A much higher); LECT2 (A: -2.38, B: 0.72, B much higher); RALB (A: -3.57, B: -0.48, B much higher); VSIG2 (A: -2.18, B: 0.87, B much higher); RNF4 (A: -1.77, B: 1.23, B much higher); FGF16 (A: -2.27, B: 0.72, B much higher); OMG (A: 0.68, B: -2.28, A much higher); FHIP2A (A: 1.44, B: -1.48, A much higher); MELTF (A: 2.13, B: -0.76, A much higher); SLC4A1 (A: -2.32, B: 0.56, B much higher); SSC4D (A: 4.29, B: 1.41, A much higher); DCTN2 (A: -3.08, B: -0.20, B much higher); NAA80 (A: -0.88, B: 1.98, B much higher); FIS1 (A: 0.88, B: -1.98, A much higher); PRL (A: -1.59, B: 1.27, B much higher); ITGB1BP1 (A: -2.53, B: 0.31, B much higher); NPY (A: -1.61, B: 1.21, B much higher); DUSP3 (A: -3.17, B: -0.39, B much higher); FGF21 (A: -0.49, B: 2.28, B much higher); MLN (A: -1.81, B: 0.96, B much higher); SHPK (A: -2.31, B: 0.42, B much higher); MAP1LC3A (A: -2.15, B: 0.54, B much higher); LATS1 (A: -3.00, B: -0.32, B much higher); NFX1 (A: -3.30, B: -0.62, B much higher); ADGRG1 (A: -0.77, B: 1.90, B much higher); PDP1 (A: 1.04, B: -1.61, A much higher)

Options: A. Patient-A B. Patient-B /no_think"""

input_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
    return_dict=False,
).to(model.device)

output = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.3,
    min_p=0.15,
    repetition_penalty=1.05,
    max_new_tokens=512,
)

print(tokenizer.decode(output[0, input_ids.shape[-1]:], skip_special_tokens=True))

B


**Adapting this to other task types.** Both single-individual forms drop the differential selection and report NPX values for one person.

* **Multiclass age (`LB-0074`)**: one proteome, assigned to an age decade.

  ```
  Which age group is this individual in? Answer with one letter.

  Options: A. 20-29  B. 30-39  C. 40-49  D. 50-59  E. 60-69  F. 70-79  G. 80+ /no_think
  ```

* **Regression age (`LB-0082`)** : one proteome, no options.

  ```
  How old is this individual? Answer with a number of years. /no_think
  ```

### Genetic evidence (OpenGenes)

This domain is structurally different. Instead of measurements from individuals, the prompt assembles curated evidence about one gene, covering animal-model expression changes, interaction partners, GO annotations, and tissue expression, then asks whether that gene's expression rises or falls with age in human muscle.

The gene is *OPN3*, masked throughout, so the model has to reason from the evidence rather than recall an answer. The animal-model observations are mixed, reporting both increases and decreases.

The expected answer for the example below is **A**, that expression decreases with age.

In [7]:
prompt = """Animal model evidence:
In the brain of C57BL/6 mice, a gene increased gene expression, as established by measuring protein levels. In the spinal cord of C57BL/6 mice, a gene decreased gene expression, as established by measuring protein levels. In the brain of rhesus monkeys between ages 5 and 31 years, a gene decreased gene expression by 43.0%, as established by measuring protein levels. In the brain of C57BL/6 × C3H mice between ages 5 and 30 months, a gene increased gene expression by 26.8%, as established by measuring protein levels. In the brain of C57BL/6 mice between ages 2 and 20 months, a gene decreased gene expression, as established by measuring mRNA levels.

PPI network:
This gene interacts with CHML and KMO in humans.

GO term annotations:
This gene has molecular functions including all-trans retinal binding, G protein-coupled photoreceptor activity, 11-cis retinal binding, G protein-coupled receptor activity, and photoreceptor activity; participates in negative regulation of apoptotic process, positive regulation of cellular respiration, cellular response to UV-A, response to blue light, and adenylate cyclase-inhibiting G protein-coupled receptor signaling pathway; is located in membrane, cytoplasm, cytosol, photoreceptor outer segment, and plasma membrane.

Protein Atlas expression:
This gene mRNA (Tau=0.81, tissue-enhanced) is most expressed in cerebral cortex and liver (nTPM 6-11). Low or not detected in all other tissues. Medium in Adrenal gland, Bronchus, Duodenum, Gallbladder, Heart muscle, Nasopharynx, Parathyroid gland, Prostate, Rectum, Salivary gland, Seminal vesicle, Skeletal muscle, Small intestine, Stomach, Thyroid gland, Urinary bladder, Appendix (mixed), Cervix (mixed), Colon (mixed), Endometrium (mixed), Kidney (mixed), Liver (mixed), Lung (mixed), Pancreas (mixed), and Skin (mixed). Low in Adipose tissue, Bone marrow, Epididymis, Esophagus, Fallopian tube, Hippocampus, Oral mucosa, Smooth muscle, Breast (mixed), Caudate (mixed), Cerebellum (mixed), Cerebral cortex (mixed), Placenta (mixed), Soft tissue (mixed), Spleen (mixed), Testis (mixed), and Tonsil (mixed). Not detected in Lymph node, Ovary, and Vagina.

In muscle of humans between ages 18 and 65 years, Gene X mRNA expression measured by microarray:

Options: A. decreases with age B. increases with age /no_think"""

input_ids = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt}],
    add_generation_prompt=True,
    return_tensors="pt",
    tokenize=True,
    return_dict=False,
).to(model.device)

output = model.generate(
    input_ids,
    do_sample=True,
    temperature=0.3,
    min_p=0.15,
    repetition_penalty=1.05,
    max_new_tokens=512,
)

print(tokenizer.decode(output[0, input_ids.shape[-1]:], skip_special_tokens=True))

A


**Adapting this to other task types.**

* **Pairwise lifespan (`LB-0122`)** : two perturbations, which extends lifespan more.

  ```
  Which gene-pair perturbation extends lifespan more? Answer with one letter.

  Options: A. Perturbation A  B. Perturbation B /no_think
  ```

* **Regression lifespan (`LB-0126`)**: one perturbation, percentage change.

  ```
  By what percentage does this gene-pair perturbation change lifespan? Answer with a number. /no_think
  ```

## References

- Release blog post: [What Can a Small Language Model Learn from Aging Data?](https://www.liquid.ai/blog/longevity)
- Paper: Zhavoronkov, A. et al. (2026). [An Open Benchmark and Language Models for AI in Aging Biology](https://doi.org/10.1016/j.cell.2026.08.026). *Cell* 189, 1–15.
- Models: [LFM2-1.2B-Longevity](https://huggingface.co/LiquidAI/LFM2-1.2B-Longevity) and [LFM2-2.6B-Longevity](https://huggingface.co/LiquidAI/LFM2-2.6B-Longevity)
- Benchmark: [LongevityBench](https://huggingface.co/datasets/insilicomedicine/longebench)
- Base models: [LFM2-1.2B](https://huggingface.co/LiquidAI/LFM2-1.2B) and [LFM2-2.6B](https://huggingface.co/LiquidAI/LFM2-2.6B)
- [LFM2 Technical Report](https://arxiv.org/abs/2511.23404)
- [Chat template documentation](https://docs.liquid.ai/lfm/key-concepts/chat-template) and [Transformers inference guide](https://docs.liquid.ai/lfm/inference/transformers)

### Need help building with our models and tools?

Join the Liquid AI Discord community and ask.

<a href="https://discord.com/invite/liquid-ai"><img src="https://img.shields.io/discord/1385439864920739850?color=7289da&label=Join%20Discord&logo=discord&logoColor=white" alt="Join Discord"></a>